In [ ]:
import numpy as np
import pandas as pd
import os
import joblib

from sklearn.preprocessing import MinMaxScaler
from snowflake.snowpark.context import get_active_session

In [ ]:
session = get_active_session()
water_data = pd.read_csv('water_quality_training_dataset.csv')
display(water_data.head(5))

In [ ]:
landsat_train_features = pd.read_csv("landsat_features_training.csv")
display(landsat_train_features.head(5))

In [ ]:
landsat_train_features['NDMI'] = landsat_train_features['NDMI'].astype(float)
landsat_train_features['MNDWI'] = landsat_train_features['MNDWI'].astype(float)

In [ ]:
Terraclimate_df = pd.read_csv("terraclimate_features_training.csv")
display(Terraclimate_df.head(5))

In [ ]:
# Combining all datasets
water_terras_df = water_data.merge(
    Terraclimate_df,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

complete_merge = water_terras_df.merge(
    landsat_train_features,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

display(complete_merge.head(5))

In [ ]:
# Verifying that all data is of the correct datatype
complete_merge.dtypes

# Converting the "Sample Data" column to the correct datatype
complete_merge["Sample Date"] = pd.to_datetime(complete_merge["Sample Date"], format= r"%d-%m-%Y")

# Verifying that the datatypes are all correct now
complete_merge.dtypes

In [ ]:
complete_merge['sensor_id'] = complete_merge['Latitude'].astype(str) + '_' + complete_merge['Longitude'].astype(str)

In [ ]:
df = complete_merge.sort_values(['sensor_id', 'Sample Date'])

# Imputing all missing data
df = df.fillna(df.median(numeric_only=True))

# Checking for any missing values
df.isna().sum()

display(df.head(5))

In [ ]:
# --- 1. Scaler Setup ---
input_scaler = MinMaxScaler()
output_scaler = MinMaxScaler()

# Separate features and targets for clarity later
weather_cols = ['swir22', 'NDMI', 'MNDWI', 'pet']
target_cols = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

# Fit/Transform (Reminder: Only fit on TRAINING data if possible!)
df[weather_cols] = input_scaler.fit_transform(df[weather_cols])
df[target_cols] = output_scaler.fit_transform(df[target_cols])

# Assuming 'scaler_weather' is your fitted MinMaxScaler
input_scale_file = "input_scaler.save"
output_scale_file = "output_scaler.save"
joblib.dump(input_scaler, input_scale_file) 
joblib.dump(output_scaler, output_scale_file)

print("Scaler saved successfully!")

# --- 2. Median Calculation (Crucial Fix) ---
# We ONLY need the medians for the weather columns for the X filler
weather_median = df[weather_cols].median().values

def build_3d_dataset(df, weather_median, time_steps=24):
    X, y = [], []
     
    for _, group in df.groupby('sensor_id'):
        weather_values = group[weather_cols].values
        target_values = group[target_cols].values
        
        for i in range(1, len(weather_values) + 1):
            # 1. Create the X (Input) window
            current_input_history = weather_values[:i]
            
            if len(current_input_history) < time_steps:
                needed = time_steps - len(current_input_history)
                filler = np.tile(weather_median, (needed, 1))
                window = np.vstack([filler, current_input_history])
            else:
                window = current_input_history[-time_steps:]
            
            # 2. Create the y (Target)
            # We want to predict the WQ at the CURRENT time step 'i-1'
            target = target_values[i-1]
            
            X.append(window)
            y.append(target)
            
    return np.array(X), np.array(y)

In [ ]:
X, y = build_3d_dataset(df, weather_median, time_steps=5)

print(f"Final Shape: {X.shape}")
print(f"Final Shape: {y.shape}")

In [ ]:
np.save('/tmp/X_train_res_3d.npy', X)
np.save('/tmp/y_train_res.npy', y)

print(f"Files saved to session: {os.listdir('.')}")

In [ ]:
session.sql(f"""
    PUT file:///tmp/X_train_res_3d.npy
    snow://workspace/USER$.PUBLIC."ey-ai_d-challenge"/versions/live/
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

session.sql(f"""
    PUT file:///tmp/y_train_res.npy
    snow://workspace/USER$.PUBLIC."ey-ai_d-challenge"/versions/live/
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print("File saved! Refresh the browser to see the files in the sidebar")